# 🔬 Breast Cancer Detection — CNN on Histopathology Images

**Binary classification of breast histopathology patches: malignant (IDC-positive) vs. benign (IDC-negative).**

This notebook trains a convolutional neural network in PyTorch to detect Invasive Ductal
Carcinoma (IDC) — the most common form of breast cancer — in histopathology image patches.

**Dataset options**
1. **Real data (recommended):** the [Breast Histopathology Images (IDC)](https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images)
   dataset or [BreaKHis](https://web.inf.ufpr.br/vri/databases/breast-cancer-histopathological-database-breakhis/).
   Point `DATA_DIR` at a folder with `benign/` and `malignant/` subfolders of PNG/JPEG patches.
2. **Built-in fallback:** if no dataset is found, the notebook **generates a synthetic
   H&E-like dataset inline** so the full pipeline runs end-to-end anywhere (same spirit as
   the other self-contained notebooks in this repo). Swap in real data for meaningful results.

**Pipeline:** data loading → EDA → augmentation → CNN training → evaluation
(confusion matrix, ROC/AUC) → inference demo → Grad-CAM interpretability.

## 1. Setup

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Dataset

### Option A — real histopathology data
Download the IDC dataset from Kaggle and arrange it as:
```
data/idc/
├── benign/     # class 0 patches (*.png)
└── malignant/  # class 1 patches (*.png)
```
> 💡 The Kaggle IDC download stores patches as `<patient>/<class>/*.png` — a short
> reorganization script is included below (commented out) if you use that layout.

### Option B — synthetic fallback (default)
If `DATA_DIR` does not exist, synthetic H&E-stained-like patches are generated:
- **Benign:** sparse, small, uniform nuclei.
- **Malignant:** dense, clustered, irregular, size-variable nuclei (pleomorphism).

In [ ]:
# Point this at your real dataset root (with benign/ and malignant/ subfolders).
DATA_DIR = Path(os.environ.get("BCD_DATA_DIR", "data/idc"))

IMG_SIZE = 96          # patch size (IDC patches are 50x50; we resize)
SYNTH_PER_CLASS = 600  # fallback dataset size per class

def find_real_images(root: Path):
    """Return (paths, labels) if root/benign and root/malignant exist, else None."""
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    benign = sorted(p for p in (root / "benign").glob("*") if p.suffix.lower() in exts) if (root / "benign").is_dir() else []
    malignant = sorted(p for p in (root / "malignant").glob("*") if p.suffix.lower() in exts) if (root / "malignant").is_dir() else []
    if benign and malignant:
        return [str(p) for p in benign + malignant], [0] * len(benign) + [1] * len(malignant)
    return None

# --- Optional: reorganize the raw Kaggle IDC layout into benign/ and malignant/ ---
# import shutil
# raw = Path("data/raw_idc")          # contains <patient_id>/<0 or 1>/*.png
# for cls, name in [("0", "benign"), ("1", "malignant")]:
#     (DATA_DIR / name).mkdir(parents=True, exist_ok=True)
#     for png in raw.glob(f"*/{cls}/*.png"):
#         shutil.copy(png, DATA_DIR / name / f"{png.parent.parent.name}_{png.name}")

In [ ]:
def _draw_nuclei(draw, n_nuclei, malignant, rng):
    """Draw cell nuclei; malignant => denser, larger, clustered, irregular."""
    for _ in range(n_nuclei):
        if malignant and rng.random() < 0.7:  # clustered growth
            cx, cy = 24 + rng.random() * 48, 24 + rng.random() * 48
        else:                                 # uniform spread
            cx, cy = rng.random() * 96, rng.random() * 96
        r = rng.uniform(3, 9) if malignant else rng.uniform(2, 5)
        if malignant and rng.random() < 0.25:
            r *= 1.8                          # enlarged pleomorphic nuclei
        shade = int(rng.uniform(70, 130))
        color = (shade, int(shade * 0.35), int(shade * 0.9))  # hematoxylin purple
        # irregular ellipse
        rx, ry = r * rng.uniform(0.7, 1.3), r * rng.uniform(0.7, 1.3)
        draw.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=color)

def make_synthetic_patch(malignant: bool, rng: random.Random) -> Image.Image:
    """Generate one synthetic H&E-like histopathology patch (96x96 RGB)."""
    base = np.full((96, 96, 3), (236, 190, 216), dtype=np.float32)      # eosin pink
    base += np.random.normal(0, 12, base.shape)                        # tissue noise
    img = Image.fromarray(np.clip(base, 0, 255).astype(np.uint8))
    draw = ImageDraw.Draw(img)
    # stroma strands
    for _ in range(rng.randint(3, 8)):
        pts = [(rng.random() * 96, rng.random() * 96) for _ in range(3)]
        draw.line(pts, fill=(220, 160, 190), width=rng.randint(2, 5))
    n_nuclei = rng.randint(80, 150) if malignant else rng.randint(25, 60)
    _draw_nuclei(draw, n_nuclei, malignant, rng)
    return img.filter(ImageFilter.GaussianBlur(0.6))

# Preview one synthetic patch of each class
_rng = random.Random(SEED)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, mal, title in zip(axes, [False, True], ["Benign (synthetic)", "Malignant (synthetic)"]):
    ax.imshow(make_synthetic_patch(mal, _rng)); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

### Build the dataset (real if available, synthetic otherwise)

In [ ]:
class HistopathologyDataset(Dataset):
    """Loads real patches from disk, or generates synthetic ones on the fly."""
    def __init__(self, paths=None, labels=None, n_synth_per_class=SYNTH_PER_CLASS,
                 transform=None, seed=SEED):
        self.transform = transform
        self.paths, self.labels = paths, labels
        self.synthetic = paths is None
        if self.synthetic:
            self.labels = [0] * n_synth_per_class + [1] * n_synth_per_class
        self._rng = random.Random(seed)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx]
        if self.synthetic:
            rng = random.Random(SEED + idx)          # deterministic per sample
            img = make_synthetic_patch(bool(label), rng)
        else:
            img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

found = find_real_images(DATA_DIR)
if found:
    paths, labels = found
    print(f"✅ Real dataset found: {len(paths)} patches "
          f"({labels.count(0)} benign / {labels.count(1)} malignant)")
    full_dataset = HistopathologyDataset(paths=paths, labels=labels)
else:
    print(f"⚠️ No dataset at '{DATA_DIR}' — using SYNTHETIC fallback data.")
    full_dataset = HistopathologyDataset()

counts = np.bincount(full_dataset.labels)
plt.figure(figsize=(4, 3))
sns.barplot(x=["Benign", "Malignant"], y=counts, palette=["#4C9AFF", "#FF5A76"])
plt.title("Class distribution"); plt.ylabel("patches"); plt.show()

### Sample patches

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, cls in enumerate([0, 1]):
    idxs = [i for i, l in enumerate(full_dataset.labels) if l == cls][:8]
    for col, i in enumerate(idxs):
        img, _ = full_dataset[i]  # untransformed PIL image
        axes[row, col].imshow(img); axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(["Benign", "Malignant"][row], loc="left", fontsize=12)
plt.suptitle("Random patches per class", y=1.02); plt.tight_layout(); plt.show()

## 3. Preprocessing, augmentation & splits

- **Train:** random flips/rotation + color jitter (stain variation).
- **Val/Test:** resize + tensor only.
- Split: **70% train / 15% val / 15% test** with a seeded shuffle.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

n = len(full_dataset)
n_train, n_val = int(0.7 * n), int(0.15 * n)
n_test = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED))

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset, self.transform = subset, transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, i):
        img, label = self.subset[i]
        return (self.transform(img) if self.transform else transforms.ToTensor()(img)), label

BATCH = 32
train_loader = DataLoader(TransformSubset(train_ds, train_tf), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TransformSubset(val_ds, eval_tf), batch_size=BATCH)
test_loader  = DataLoader(TransformSubset(test_ds, eval_tf), batch_size=BATCH)
print(f"train={n_train}  val={n_val}  test={n_test}  | batches/epoch={len(train_loader)}")

## 4. Model

A compact CNN built from scratch (~1.1M params) — good for 96×96 patches.
Set `USE_TRANSFER = True` to fine-tune a pretrained ResNet-18 instead.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.4), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

USE_TRANSFER = False
if USE_TRANSFER:
    model = models.resnet18(weights="DEFAULT")
    model.fc = nn.Linear(model.fc.in_features, 2)
else:
    model = SimpleCNN()
model = model.to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, preds, targets = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            preds += out.argmax(1).tolist(); targets += y.tolist()
    return total_loss / len(targets), accuracy_score(targets, preds)

EPOCHS = 5
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val, best_state = 0.0, None
for ep in range(1, EPOCHS + 1):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader, criterion)
    scheduler.step()
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl);   history["val_acc"].append(va)
    if va > best_val:
        best_val, best_state = va, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {ep}/{EPOCHS} | train loss {tl:.4f} acc {ta:.3f} | val loss {vl:.4f} acc {va:.3f}")

model.load_state_dict(best_state)  # restore best checkpoint
print(f"Best val accuracy: {best_val:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Evaluation on the held-out test set

In [ ]:
model.eval()
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        probs = F.softmax(model(x.to(DEVICE)), dim=1)[:, 1].cpu().numpy()
        y_prob += probs.tolist(); y_true += y.tolist()
        y_pred += (probs >= 0.5).astype(int).tolist()

print(f"Test accuracy: {accuracy_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, target_names=["benign", "malignant"], digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["benign", "malignant"], yticklabels=["benign", "malignant"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix"); plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(5, 4.5))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curve"); plt.legend(); plt.show()

## 7. Inference demo

Random test patches with predicted class and malignant probability.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
sample_idx = random.sample(range(len(test_ds)), 12)
wrapped = TransformSubset(test_ds, eval_tf)
for ax, i in zip(axes.flat, sample_idx):
    img_t, label = wrapped[i]
    with torch.no_grad():
        p = F.softmax(model(img_t.unsqueeze(0).to(DEVICE)), 1)[0, 1].item()
    pred = int(p >= 0.5)
    img = img_t.permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(np.clip(img, 0, 1))
    color = "green" if pred == label else "red"
    ax.set_title(f"true={['ben','mal'][label]} pred={['ben','mal'][pred]}\np(mal)={p:.2f}",
                 fontsize=9, color=color)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 8. Grad-CAM — where is the model looking?

Grad-CAM on the last conv block highlights the regions driving the prediction
(ideally the nuclei clusters, which match the pathologist's criteria).

In [ ]:
def grad_cam(model, img_tensor, target_layer):
    model.eval()
    activations, gradients = {}, {}
    def fwd_hook(m, i, o): activations["feat"] = o
    def bwd_hook(m, gi, go): gradients["grad"] = go[0]
    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)
    out = model(img_tensor.unsqueeze(0).to(DEVICE))
    model.zero_grad()
    out[0, out.argmax(1)].backward()
    h1.remove(); h2.remove()
    w = gradients["grad"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * activations["feat"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

target_layer = model.features[-1] if isinstance(model, SimpleCNN) else model.layer4[-1]
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for col, i in enumerate(random.sample(range(len(test_ds)), 4)):
    img_t, label = wrapped[i]
    cam = grad_cam(model, img_t, target_layer)
    img = np.clip(img_t.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1)
    cam_img = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0
    axes[0, col].imshow(img); axes[0, col].set_title(f"patch (true={['benign','malignant'][label]})"); axes[0, col].axis("off")
    axes[1, col].imshow(img); axes[1, col].imshow(cam_img, cmap="jet", alpha=0.45)
    axes[1, col].set_title("Grad-CAM overlay"); axes[1, col].axis("off")
plt.tight_layout(); plt.show()

## 9. Save the model

In [ ]:
torch.save(model.state_dict(), "breast_cancer_cnn.pt")
print("Saved to breast_cancer_cnn.pt")

## ✅ Summary & next steps

- End-to-end deep learning pipeline for breast cancer (IDC) detection in histopathology patches.
- **For real results**, drop the IDC/BreaKHis data into `data/idc/{benign,malignant}/` and re-run.
- Ideas to improve: `USE_TRANSFER = True` (ResNet-18 fine-tuning), stain normalization
  (Macenko), class-weighted loss for imbalance, test-time augmentation, and external
  validation on a different hospital's slides.

> ⚠️ **Disclaimer:** this is an educational project — not a medical device. Real clinical
> deployment needs regulatory validation, calibration, and pathologist oversight.